In [1]:
import radiate as rd
import polars as pl
from IPython.display import display, HTML

rd.random.seed(33)

In [2]:
def compute(x: float) -> float:
    return 4.0 * x**3 - 3.0 * x**2 + x


inputs = []
answers = []

input = -1.0
for _ in range(-10, 10):
    input += 0.1
    inputs.append([input])
    answers.append([compute(input)])

In [3]:
collector = rd.MetricCollector()

wait = (
    rd.Expr.when(rd.Expr.generation() > 10)
    .then(rd.Expr.select("scores.best").rolling(2).slope())
    .otherwise(0)
)

slope = rd.Expr.select("scores.best").rolling(5).slope() * 0.02

engine = (
    rd.Engine.graph(
        shape=(1, 1),
        vertex=[rd.Op.sub(), rd.Op.mul(), rd.Op.sum()],
        edge=rd.Op.weight(),
        output=rd.Op.sum(),
    )
    .regression(inputs, answers, loss=rd.MSE)
    .subscribe(collector)
    .metrics(slope=slope)
    .alter(
        rd.Cross.graph(0.05, 0.5),
        rd.Mutate.op(0.07, 0.05),
        rd.Mutate.graph(0.1, 0.1, False),
    )
    .limit(rd.Limit.score(0.001), rd.Limit.generations(1000))
)

result = engine.run(log=True)

2026-10-17T12:16:03.110700Z  INFO State Change: PreStart -> Running
2026-10-17T12:16:03.111217Z  INFO Epoch 1    | Score:   2.9954 | Time: 176.17µs
2026-10-17T12:16:03.111344Z  INFO Epoch 2    | Score:   2.0038 | Time: 265.33µs
2026-10-17T12:16:03.111441Z  INFO Epoch 3    | Score:   2.0038 | Time: 337.00µs
2026-10-17T12:16:03.111501Z  INFO Epoch 4    | Score:   2.0038 | Time: 382.71µs
2026-10-17T12:16:03.111560Z  INFO Epoch 5    | Score:   2.0038 | Time: 429.79µs
2026-10-17T12:16:03.111612Z  INFO Epoch 6    | Score:   2.0038 | Time: 465.54µs
2026-10-17T12:16:03.111661Z  INFO Epoch 7    | Score:   1.6821 | Time: 503.67µs
2026-10-17T12:16:03.111721Z  INFO Epoch 8    | Score:   1.6821 | Time: 551.04µs
2026-10-17T12:16:03.111774Z  INFO Epoch 9    | Score:   1.6821 | Time: 590.79µs
2026-10-17T12:16:03.111831Z  INFO Epoch 10   | Score:   1.6821 | Time: 635.00µs
2026-10-17T12:16:03.111888Z  INFO Epoch 11   | Score:   1.6821 | Time: 679.00µs
2026-10-17T12:16:03.111944Z  INFO Epoch 12   | Score

In [4]:
collector.plot("scores.best", "pct.diversity")
# print(result.metrics().dashboard())

In [6]:
eval_results = result.value.eval(inputs)
accuracy = rd.accuracy(result.value, inputs, answers, loss=rd.MSE)
accuracy

PyAccuracy("Regression Accuracy" {
	N: 20 
	Accuracy: 99.45%
	R² Score: 0.99923
	RMSE: 0.05613
	Loss (MSE): 0.00315
})

In [7]:
df = collector.to_polars(lazy=False)
df

name,last,sum,mean,stddev,var,skew,min,max,count,time_sum,time_mean,time_stddev,time_min,time_max,time_var,generation,update_count,tags
str,f64,f64,f64,f64,f64,f64,f64,f64,i64,duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],i64,i64,list[str]
"""count.evaluation""",9.0,109.0,54.5,64.346718,4140.5,NaN,9.0,100.0,2,null,null,null,null,null,null,0,2,"[""statistic""]"
"""step.evaluate.time""",0.000012,0.000071,0.000036,0.000034,1.1460e-9,NaN,0.000012,0.000059,2,71µs,35µs,33µs,11µs,59µs,0µs,0,2,"[""time"", ""step""]"
"""selector.roulette""",20.0,20.0,20.0,0.0,0.0,NaN,20.0,20.0,1,null,null,null,null,null,null,0,1,"[""selector"", ""statistic""]"
"""selector.roulette.time""",0.000006,0.000006,0.000006,0.0,0.0,NaN,0.000006,0.000006,1,6µs,6µs,0µs,6µs,6µs,0µs,0,1,"[""selector"", ""time""]"
"""selector.tournament""",80.0,80.0,80.0,0.0,0.0,NaN,80.0,80.0,1,null,null,null,null,null,null,0,1,"[""selector"", ""statistic""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""mutator.graph.rate.edge""",0.1,100.0,0.1,0.0,0.0,0.0,0.1,0.1,1000,null,null,null,null,null,null,999,1,"[""alterer"", ""mutator"", … ""expr""]"
"""step.metric.time""",0.000008,0.008837,0.000009,0.000002,3.4927e-12,0.0,0.000007,0.000041,1000,8837µs,8µs,1µs,7µs,40µs,0µs,999,1,"[""time"", ""step""]"
"""time""",0.000147,0.113468,0.000113,0.000033,1.0797e-9,0.0,0.000034,0.000299,1000,113467µs,113µs,32µs,33µs,299µs,0µs,999,1,"[""time""]"


In [ ]:
filtered = (
    df.filter(pl.col("name") == "score.improvement")
    .select("generation")
    .unique()
    .sort("generation")
)
filtered

generation
i64
0
1
2
3
4
…
995
996
997


In [ ]:
display(HTML(filtered._repr_html_()))

generation
i64
0
1
2
3
4
…
995
996
997
